# VQA 앙상블 — 리더보드 0.97140

한국어 장면 텍스트 4지선다. 학습 뷰 5개와 무학습 뷰 2개를 계열 평균

## 최종 구성

| 계열 | 뷰 | 모델 | 학습 | 설정 | 단독 검증 | 지분 |
|---|---|---|---|---|---|---|
| **Q** | `Q` | Qwen3-VL-8B | LoRA r32 | 768토큰, TTA 2 | 0.9550 | 1/9 |
| | `Q2` | 〃 (같은 어댑터) | 〃 | 1024토큰, TTA 1 | 0.9667 | 1/9 |
| | `Q3` | 〃 (같은 어댑터) | 〃 | 1280토큰, TTA 1 | 0.9650 | 1/9 |
| **HR** | `HR` | Qwen3-VL-4B | LoRA r32 | 1280토큰, TTA 2 | 0.9550 | 1/9 |
| | `HR2` | 〃 (같은 어댑터) | 〃 | 1024토큰, TTA 1 | 0.9567 | 1/9 |
| | `VC` | VARCO-VISION-2.0-14B | 없음 | AnyRes 2배, TTA 1 | 0.8967 | 1/9 |
| **G** | `G26B` | gemma-4-26B-A4B-it | 없음 | 560토큰, TTA 2 + 표기 프롬프트 | 0.9583 | 1/3 |

앙상블 검증 **0.9733**, 리더보드 **0.97140**. 최고 단일 뷰(`Q2` 0.9667)보다 3문항 높음

## 점수 변화

| 구성 | 리더보드 | 증분 |
|---|---|---|
| Qwen 5뷰 (Q·Q2·Q3·HR·HR2) | 0.965 | — |
| + G26 | 0.970 | +0.005 |
| + VC | 0.97100 | +0.001 |
| G26 → G26B | **0.97140** | +0.0004 |

---
# 공통 (모든 단계에서 실행)

| 셀 | 내용 | 산출물 |
|---|---|---|
| 1 | `PHASE` 에 맞는 transformers 버전 설치 | — |
| 2 | 드라이브 마운트, 데이터 압축 해제 (zip 해시로 캐시) | `train.csv` `test.csv` |
| 3 | 시드·LoRA 하이퍼파라미터·모델별 설정 | `CFG` `OUT_DIR` |
| 4 | md5 그룹 분할로 검증 600행, `dev.csv` 다수결 2,097행 학습 추가 | `va_df` `test_df` `val_y.npy` |
| 4.5 | 전역 `nn.Module` 전수 해제, HF 캐시 삭제 함수 | `free_gpu` `drop_models` |

In [ ]:
# ============================== 1. 설치 ==============================
# 단계를 바꿀 때마다 여기만 고치고 1 → 2 → 3 → 4 순서로 다시 실행하세요.
PHASE = "internvl"   # "qwen" / "gemma" / "varco" / "internvl"

import subprocess, sys
PKGS = {
    "qwen":     ["transformers>=4.57.3", "peft>=0.14", "accelerate", "pyzipper"],
    "gemma":    ["transformers>=5.17", "accelerate", "pyzipper"],   # Gemma 4 는 5.17 이상 필요
    "varco":    ["transformers>=5.17", "accelerate", "pyzipper"],   # LLaVA-OneVision 계열
    "internvl": ["transformers>=5.17", "accelerate", "pyzipper"],   # InternVL3 -hf 변환판
}[PHASE]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS], check=True)

import importlib; importlib.invalidate_caches()
import torch, transformers
cap = torch.cuda.get_device_capability(0)
print(f"PHASE={PHASE} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"{torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB | sm_{cap[0]}{cap[1]}")
assert torch.cuda.is_available(), "런타임 유형을 GPU 로 바꾸세요"
assert torch.cuda.is_bf16_supported(), "bf16 미지원 GPU 입니다. 다른 런타임을 선택하세요"
if PHASE != "qwen":
    ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    if ver < (5, 17): print("⚠ 버전이 낮습니다 → 런타임 › 세션 다시 시작 후 2번 셀부터 실행하세요")
    else:             print("버전 확인 완료 — 재시작 없이 2번 셀로 진행해도 됩니다")
else:
    print("설치 완료 — 버전이 예상과 다르면 세션을 다시 시작하세요")

In [ ]:
# ============================== 2. 드라이브 · 데이터 압축 해제 ==============================
ZIP_PATH     = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"
EXP_DIR      = "/content/drive/MyDrive/VQA_runs/qwen3vl_hr1280"   # 기존 뷰가 저장된 폴더
DATA_BASE    = "/content/vqa_data"

import os, sys, json, shutil, hashlib
from pathlib import Path, PurePosixPath
from google.colab import drive
drive.mount("/content/drive")

if not Path(ZIP_PATH).is_file():
    cand = sorted(p for p in Path("/content/drive/MyDrive").rglob("*.zip") if p.stat().st_size > 10**7)[:20]
    raise FileNotFoundError(f"ZIP 이 없습니다: {ZIP_PATH}\n드라이브에서 찾은 후보:\n" +
                            ("\n".join(f"  {p}" for p in cand) or "  (없음)"))

import pyzipper
from tqdm.auto import tqdm

h = hashlib.sha256()
with open(ZIP_PATH, "rb") as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b""): h.update(block)
DATA_ROOT = Path(DATA_BASE) / h.hexdigest()[:16]
marker = DATA_ROOT / "extraction_complete.json"

if marker.exists():
    print("이미 풀린 데이터 재사용:", DATA_ROOT)
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with pyzipper.AESZipFile(ZIP_PATH) as z:
        z.setpassword(ZIP_PASSWORD.encode())
        infos = z.infolist()
        for info in infos:
            p = PurePosixPath(info.filename.replace("\\", "/"))
            if p.is_absolute() or ".." in p.parts: raise ValueError(f"안전하지 않은 경로: {info.filename}")
        need = sum(i.file_size for i in infos)
        assert shutil.disk_usage(DATA_ROOT).free > need + 5 * 1024**3, "로컬 디스크 공간 부족"
        for info in tqdm(infos, desc="압축 해제"):
            target = DATA_ROOT.joinpath(*PurePosixPath(info.filename.replace("\\", "/")).parts)
            if info.is_dir(): target.mkdir(parents=True, exist_ok=True); continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as s, open(target, "wb") as d: shutil.copyfileobj(s, d, 1024 * 1024)
    marker.write_text("{}")
    print("압축 해제 완료:", DATA_ROOT)

hits = sorted(p.parent for p in DATA_ROOT.rglob("train.csv") if "__MACOSX" not in p.parts)
assert hits, f"train.csv 를 찾지 못했습니다: {DATA_ROOT}"
os.chdir(hits[0])
Path(EXP_DIR).mkdir(parents=True, exist_ok=True)
print("작업 폴더:", os.getcwd(), "| 결과 폴더:", EXP_DIR)

In [ ]:
# ============================== 3. 설정 ==============================
RUN = "Q"          # A단계에서만 사용. "HR" -> Qwen3-VL-4B / "Q" -> Qwen3-VL-8B

SEED       = 42
SPLIT_SEED = 42    # 모든 뷰가 같은 검증셋을 쓰도록 고정. 바꾸지 마세요
N_VALID    = 600
LR         = 1e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
WARMUP     = 0.05
OUT_DIR    = EXP_DIR

# --- dev.csv 를 학습 데이터로 추가 (answer1~5 다수결) ---
USE_DEV        = True
DEV_MIN_VOTES  = 3
DEV_MIN_AGREE  = 0.5
DEV_MIN_MARGIN = 1

# --- 리사이즈한 이미지 캐시 (Qwen 뷰에서만 사용) ---
CACHE_DIR  = "/content/imgcache"
CACHE_Q    = 95

CFG = {
    "Q":  dict(MODEL_ID="Qwen/Qwen3-VL-8B-Instruct", QUANT_4BIT=False,
               BASE_TOK=768, COUNT_MULT=1.0, EPOCHS=1, BATCH=4, ACCUM=2, INFER_BATCH=16, TTA=2),
    "HR": dict(MODEL_ID="Qwen/Qwen3-VL-4B-Instruct", QUANT_4BIT=False,
               BASE_TOK=1280, COUNT_MULT=1.0, EPOCHS=2, BATCH=4, ACCUM=4, INFER_BATCH=32, TTA=2),
}[RUN]
GRAD_CKPT = False
RESUME    = True
TTA = CFG["TTA"]
TRAIN_MAX_MIN = 0
os.makedirs(OUT_DIR, exist_ok=True)
print(RUN, CFG)

In [ ]:
# ============================== 4. 데이터 · 검증셋 ==============================
import os, math, time, random, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "GPU(CUDA)가 필요합니다"
Image.MAX_IMAGE_PIXELS = None
torch.backends.cuda.matmul.allow_tf32 = True

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

CH  = ["a", "b", "c", "d"]
C2I = {c: i for i, c in enumerate(CH)}

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv").reset_index(drop=True)
sub_df   = pd.read_csv("sample_submission.csv")
train_df["_y"] = train_df["answer"].astype(str).str.strip().str.lower().map(C2I)
assert train_df["_y"].notna().all(), "answer 열에 a/b/c/d 외의 값이 있습니다"
train_df["_y"] = train_df["_y"].astype(int)

def qbucket(q):
    q = str(q)
    if ("몇" in q) or ("개입" in q) or ("개수" in q): return "count"
    if "재질" in q: return "material"
    if "색" in q:   return "color"
    if "종류" in q: return "kind"
    return "other"
for d in (train_df, test_df):
    d["_qb"] = d["question"].map(qbucket)

# 같은 사진이 여러 행에 쓰이므로 사진 내용(md5) 단위로 검증셋을 뗍니다 (누수 방지)
def file_md5(p):
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()
train_df["_md5"] = [file_md5(p) for p in tqdm(train_df["path"], desc="md5")]
cnt = train_df["_md5"].value_counts()
groups = np.array(sorted(cnt.index))
np.random.RandomState(SPLIT_SEED).shuffle(groups)
val_g, n = set(), 0
for g in groups:
    if n >= N_VALID: break
    val_g.add(g); n += int(cnt[g])
is_val = train_df["_md5"].isin(val_g)
tr_df = train_df[~is_val].reset_index(drop=True)
va_df = train_df[is_val].reset_index(drop=True)
y_va  = va_df["_y"].values
print(f"train {len(tr_df)} | valid {len(va_df)} | test {len(test_df)}")

old_y = Path(OUT_DIR) / "val_y.npy"
if old_y.exists():
    assert np.array_equal(np.load(old_y), y_va), "기존 뷰와 검증셋이 다릅니다. SPLIT_SEED/N_VALID 확인"
    print("기존 검증셋과 동일 확인")
else:
    np.save(old_y, y_va); print("val_y.npy 저장")

# ---- dev.csv 를 answer1~5 다수결로 학습에 추가 (A단계 학습용) ----
COLS = ["id", "path", "question", "a", "b", "c", "d", "_y", "_qb"]
tr_df = tr_df[COLS].copy()

def resolve_votes(df):
    vcols = [c for c in df.columns if c.startswith("answer") and c != "answer"]
    assert vcols, "dev.csv 에 answer1~N 열이 없습니다"
    ys, keep, drop = [], [], {"few": 0, "weak": 0, "tie": 0}
    for i, r in df.iterrows():
        v = [str(r[c]).strip().lower() for c in vcols]
        v = [x for x in v if x in C2I]
        if len(v) < DEV_MIN_VOTES: drop["few"] += 1; continue
        cnt = pd.Series(v).value_counts()
        top, second = int(cnt.iloc[0]), int(cnt.iloc[1]) if len(cnt) > 1 else 0
        if top / len(v) < DEV_MIN_AGREE: drop["weak"] += 1; continue
        if top - second < DEV_MIN_MARGIN: drop["tie"] += 1; continue
        keep.append(i); ys.append(C2I[str(cnt.index[0])])
    out = df.loc[keep].copy(); out["_y"] = ys
    print(f"dev 다수결: 채택 {len(out)} / 폐기 {drop}")
    return out.reset_index(drop=True)

if USE_DEV and os.path.exists("dev.csv"):
    dev_df = resolve_votes(pd.read_csv("dev.csv"))
    dev_df["_qb"] = dev_df["question"].map(qbucket)
    dev_df["_md5"] = [file_md5(p) for p in tqdm(dev_df["path"], desc="dev md5")]
    leak = dev_df["_md5"].isin(val_g)
    dev_df = dev_df[~leak].reset_index(drop=True)
    tr_df = pd.concat([tr_df, dev_df[COLS]], ignore_index=True)
    print(f"dev 추가 {len(dev_df)}행 (검증 누수 제외 {int(leak.sum())}행) -> 학습 {len(tr_df)}행")
else:
    print("dev 미사용")

def acc_of(p, y):
    return float((np.asarray(p).argmax(1) == np.asarray(y)).mean())

def write_sub(probs, name):
    idc, ansc = sub_df.columns[0], sub_df.columns[1]
    pred = dict(zip(test_df["id"], [CH[i] for i in np.asarray(probs).argmax(1)]))
    sub = sub_df[[idc]].copy()
    sub[ansc] = sub[idc].map(pred)
    assert sub[ansc].notna().all(), "test.csv 와 sample_submission 의 id 가 맞지 않습니다"
    sub.to_csv(name, index=False); shutil.copy(name, f"{EXP_DIR}/{name}")
    dist = (sub[ansc].value_counts(normalize=True).sort_index() * 100).round(1).to_dict()
    print(f"{name} 저장 {len(sub)}행 | 분포(%) {dist}")

In [ ]:
# ============================== 4.5 GPU · 디스크 정리 (필요할 때만) ==============================
import gc, os, glob, shutil, torch
import torch.nn as nn

def free_gpu(verbose=True):
    """전역에 남아있는 모든 nn.Module 을 찾아서 내린다."""
    g, dropped = globals(), []
    for k in list(g.keys()):
        if k.startswith("_"): continue
        try:
            if isinstance(g[k], nn.Module): dropped.append(k); del g[k]
        except Exception: pass
    for d in ["G", "V", "C", "R", "VST", "IST"]:
        if d in g and isinstance(g[d], dict):
            for k in list(g[d]):
                if isinstance(g[d][k], nn.Module): g[d][k] = None
            if "id" in g[d]: g[d]["id"] = None; dropped.append(f"{d}[dict]")
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    if verbose:
        print(f"정리: {dropped or '없음'} | VRAM {torch.cuda.memory_allocated()/1024**3:.1f} GB")

HUB = "/root/.cache/huggingface/hub"

def disk_report():
    print(f"디스크 여유 {shutil.disk_usage('/').free/1024**3:.1f} GB\n--- HF 캐시 ---")
    for d in sorted(glob.glob(f"{HUB}/models--*")):
        sz = sum(os.path.getsize(os.path.join(r, x))
                 for r, _, fs in os.walk(d) for x in fs if os.path.exists(os.path.join(r, x)))
        print(f"  {sz/1024**3:6.1f} GB  {os.path.basename(d)}")

def drop_models(*names):
    """예: drop_models('google--gemma-4-31B-it', 'NCSOFT--VARCO-VISION-2.0-14B')"""
    for n in names:
        p = f"{HUB}/models--{n}"
        if os.path.isdir(p): shutil.rmtree(p, ignore_errors=True); print("삭제:", n)
        else: print("없음:", n)
    for p in ["/root/.cache/huggingface/xet"]:
        if os.path.isdir(p): shutil.rmtree(p, ignore_errors=True); print("삭제:", p)
    print(f"디스크 여유 {shutil.disk_usage('/').free/1024**3:.1f} GB")

free_gpu(); disk_report()

---
# A. Qwen3-VL 뷰 — `PHASE = "qwen"`

주력 계열. 하나의 LoRA 어댑터로 해상도만 바꿔 뷰 3개를 만들었습니다.

| 셀 | 내용 |
|---|---|
| 5 | 모델 로드 + LoRA 부착. 시각 모듈 제외, attention·MLP 에만 r32/α64 |
| 6 | 프롬프트 조립, 해상도별 리사이즈·캐시, 배치 채점 |
| 7 | LoRA 학습. 어댑터가 있으면 건너뜀 |
| 8 | 기본 뷰 생성 (`RUN` 값에 따라 `Q` 또는 `HR`) |
| 9 | 해상도 뷰 생성 (`VIEW_TOK`·`VIEW_TAG` 지정) |



In [ ]:
# ============================== 5. Qwen 모델 로드 + LoRA ==============================
from transformers import (AutoProcessor, AutoModelForImageTextToText,
                          BitsAndBytesConfig, get_cosine_schedule_with_warmup)
from peft import LoraConfig, get_peft_model

processor = AutoProcessor.from_pretrained(CFG["MODEL_ID"])
tokenizer = processor.tokenizer
tokenizer.padding_side = "left"          # 마지막 자리 = 정답 글자가 나올 자리

ip = processor.image_processor
PS = ip.patch_size * ip.merge_size       # 시각 토큰 1개가 덮는 픽셀 (Qwen3-VL: 16 x 2 = 32)
MIN_PX, MAX_PX = 4 * PS * PS, 4096 * PS * PS
ip.min_pixels, ip.max_pixels = MIN_PX, MAX_PX
if isinstance(getattr(ip, "size", None), dict):
    ip.size = {"shortest_edge": MIN_PX, "longest_edge": MAX_PX}

kw = dict(dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
if CFG["QUANT_4BIT"]:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual", "lm_head"])
model = AutoModelForImageTextToText.from_pretrained(CFG["MODEL_ID"], **kw)
if GRAD_CKPT:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

lcfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    target_modules=r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$")
model = get_peft_model(model, lcfg)
model.print_trainable_parameters()

LETTER_IDS = [tokenizer.encode(c, add_special_tokens=False) for c in CH]
assert all(len(t) == 1 for t in LETTER_IDS), f"a/b/c/d 가 1토큰이 아닙니다: {LETTER_IDS}"
LETTER_IDS = [t[0] for t in LETTER_IDS]
print(f"PS={PS} | VRAM {torch.cuda.memory_allocated()/1024**3:.1f} GB 사용")

In [ ]:
# ============================== 6. Qwen 프롬프트 · 채점 함수 ==============================
SYSTEM = ("You are a helpful visual question answering assistant. "
          "Answer using exactly one letter among a, b, c, or d. No explanation.")

def build_text(row, perm):
    opts  = [str(row[c]).strip() for c in CH]
    shown = [opts[perm[j]] for j in range(4)]      # 화면 j번째 자리에 원래 보기 perm[j]
    user = (f"{str(row['question']).strip()}\n"
            f"(a) {shown[0]}\n(b) {shown[1]}\n(c) {shown[2]}\n(d) {shown[3]}\n\n"
            "정답을 a, b, c, d 중 소문자 한 글자로만 출력하세요.")
    msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user}]}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def tokens_for(qb):
    return int(CFG["BASE_TOK"] * (CFG["COUNT_MULT"] if qb == "count" else 1.0))

def cache_path(path, tok):
    if not CACHE_DIR: return None
    name = str(path).replace("\\", "/").strip("./").replace("/", "__")
    return os.path.join(CACHE_DIR, str(tok), name if name.lower().endswith((".jpg", ".jpeg")) else name + ".jpg")

def load_image(path, qb):
    tok = tokens_for(qb)
    cp = cache_path(path, tok)
    if cp and os.path.exists(cp):
        try:
            return Image.open(cp).convert("RGB")
        except Exception:
            pass
    im = Image.open(path).convert("RGB")
    w, h = im.size
    s = math.sqrt(tok * PS * PS / max(w * h, 1))
    nw = max(PS, int(round(w * s / PS)) * PS)
    nh = max(PS, int(round(h * s / PS)) * PS)
    im = im.resize((nw, nh), Image.BICUBIC)
    if cp:
        try:
            os.makedirs(os.path.dirname(cp), exist_ok=True)
            im.save(cp, quality=CACHE_Q, subsampling=0)
        except Exception:
            pass
    return im

def letter_logits(texts, images):
    enc = processor(text=texts, images=images, padding=True, return_tensors="pt").to("cuda")
    out = model(**enc, use_cache=False, logits_to_keep=1)
    return out.logits[:, -1, LETTER_IDS].float()   # (B, 4) : 화면 자리 a/b/c/d 점수

def make_batches(df, bs, shuffle, seed=0):
    rng = np.random.RandomState(seed)
    batches = []
    for _, idx in df.groupby("_qb").indices.items():   # 같은 유형끼리 묶어 패딩 낭비 줄임
        idx = np.array(idx)
        if shuffle: rng.shuffle(idx)
        batches += [idx[i:i + bs] for i in range(0, len(idx), bs)]
    if shuffle: rng.shuffle(batches)
    return batches

@torch.no_grad()
def predict(df, n_tta=TTA, desc="predict", save_path=None):
    model.eval()
    probs = np.zeros((len(df), 4), np.float32)
    done  = np.zeros(len(df), bool)
    if save_path and os.path.exists(save_path):
        z = np.load(save_path); probs, done = z["probs"], z["done"]
        print(f"이어서 진행: {int(done.sum())}/{len(df)}")
    batches = [b for b in make_batches(df, CFG["INFER_BATCH"], False) if not done[b].all()]
    for k, bi in enumerate(tqdm(batches, desc=desc)):
        rows = [df.iloc[j] for j in bi]
        ims  = [load_image(r["path"], r["_qb"]) for r in rows]
        agg  = torch.zeros(len(rows), 4, device="cuda")
        for s in range(n_tta):
            perm = [(j + s) % 4 for j in range(4)]
            lp = F.log_softmax(letter_logits([build_text(r, perm) for r in rows], ims), -1)
            agg[:, perm] += lp                     # 화면 자리 점수 -> 원래 보기 자리로 되돌림
        probs[bi] = F.softmax(agg / n_tta, -1).cpu().numpy()
        done[bi] = True
        if save_path and (k % 50 == 49 or k == len(batches) - 1):
            np.savez(save_path, probs=probs, done=done)
    return probs

In [ ]:
# ============================== 7. Qwen LoRA 학습 ==============================
PARTIAL = f"{OUT_DIR}/testpartial_{RUN}.npz"

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
n_batches = len(make_batches(tr_df, CFG["BATCH"], False))
total = CFG["EPOCHS"] * math.ceil(n_batches / CFG["ACCUM"])
sched = get_cosine_schedule_with_warmup(opt, int(WARMUP * total), total)
rng = np.random.RandomState(SEED)

ADAPTER = f"{OUT_DIR}/lora_{RUN}"
best_acc, best_state = -1.0, None
T_START, stop = time.time(), False

skip_training = False
if RESUME and os.path.exists(f"{ADAPTER}/adapter_model.safetensors"):
    try:
        from safetensors.torch import load_file
        from peft import set_peft_model_state_dict
        set_peft_model_state_dict(model, load_file(f"{ADAPTER}/adapter_model.safetensors"))
        skip_training = True
        print(f"기존 LoRA 를 불러왔습니다 -> {ADAPTER} (학습 건너뜀). 다시 학습하려면 RESUME=False)")
    except Exception as e:
        print("LoRA 불러오기 실패, 처음부터 학습합니다:", e)

if not skip_training and os.path.exists(PARTIAL):
    os.remove(PARTIAL)

for ep in (range(CFG["EPOCHS"]) if not skip_training else []):
    model.train()
    batches = make_batches(tr_df, CFG["BATCH"], True, SEED + ep)
    opt.zero_grad(set_to_none=True)
    run, t0 = 0.0, time.time()
    pbar = tqdm(batches, desc=f"epoch {ep+1}/{CFG['EPOCHS']}")
    for i, bi in enumerate(pbar):
        rows  = [tr_df.iloc[j] for j in bi]
        perms = [[int(x) for x in rng.permutation(4)] for _ in rows]
        y = torch.tensor([p.index(int(r["_y"])) for p, r in zip(perms, rows)], device="cuda")
        logits = letter_logits([build_text(r, p) for r, p in zip(rows, perms)],
                               [load_image(r["path"], r["_qb"]) for r in rows])
        loss = F.cross_entropy(logits, y)
        (loss / CFG["ACCUM"]).backward()
        run = loss.item() if i == 0 else 0.98 * run + 0.02 * loss.item()
        if (i + 1) % CFG["ACCUM"] == 0 or i + 1 == len(batches):
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        if i % 20 == 0: pbar.set_postfix(loss=f"{run:.4f}")
        if TRAIN_MAX_MIN and (time.time() - T_START) / 60 > TRAIN_MAX_MIN and (i + 1) % CFG["ACCUM"] == 0:
            print(f"시간 제한 {TRAIN_MAX_MIN}분 도달 -> 학습 중단"); stop = True; break
    va = predict(va_df, n_tta=1, desc="valid")
    a = acc_of(va, y_va)
    print(f"epoch {ep+1}: valid acc {a:.4f} ({(time.time()-t0)/60:.1f}분)")
    if a > best_acc:
        best_acc = a
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if "lora_" in k}
        model.save_pretrained(ADAPTER)
        print(f"  체크포인트 저장 -> {ADAPTER}")
    if stop: break

if not skip_training:
    model.load_state_dict(best_state, strict=False)
    model.save_pretrained(ADAPTER)
    print(f"최고 검증 정확도 {best_acc:.4f} 로 복원 -> {ADAPTER}")

In [ ]:
# ============================== 8. Qwen 기본 뷰 (Q 또는 HR) ==============================
va_probs = predict(va_df, desc="valid(TTA)")
print(f"[{RUN}] 검증 정확도 (TTA {TTA}) : {acc_of(va_probs, y_va):.4f}")
for b in sorted(va_df["_qb"].unique()):
    m = (va_df["_qb"] == b).values
    print(f"  {b:9s} n={m.sum():4d} acc={acc_of(va_probs[m], y_va[m]):.3f}")
np.save(f"{OUT_DIR}/valprobs_{RUN}.npy", va_probs)

test_probs = predict(test_df, desc="test", save_path=PARTIAL)
np.save(f"{OUT_DIR}/testprobs_{RUN}.npy", test_probs)
write_sub(test_probs, f"sub_{RUN}.csv")

In [ ]:
# ============================== 9. Qwen 해상도 뷰 (같은 LoRA, 다른 해상도) ==============================
# RUN="HR" 이면 VIEW_TAG="HR2"(1024), RUN="Q" 면 "Q2"(1024) / "Q3"(1280) 처럼 씁니다.
VIEW_TOK, VIEW_TAG, VIEW_BATCH = 1024, "Q2", 16

CFG["BASE_TOK"], CFG["COUNT_MULT"], CFG["INFER_BATCH"] = VIEW_TOK, 1.0, VIEW_BATCH
va2 = predict(va_df, n_tta=1, desc=f"valid {VIEW_TAG}")
print(f"[{VIEW_TAG}] {VIEW_TOK}토큰 검증 {acc_of(va2, y_va):.4f}")
np.save(f"{OUT_DIR}/valprobs_{VIEW_TAG}.npy", va2)

te2 = predict(test_df, n_tta=1, desc=f"test {VIEW_TAG}",
              save_path=f"{OUT_DIR}/testpartial_{VIEW_TAG}.npz")
np.save(f"{OUT_DIR}/testprobs_{VIEW_TAG}.npy", te2)
print("저장 완료")

---
# B. Gemma 4 뷰 — `PHASE = "gemma"` (학습 없음)

| 셀 | 내용 |
|---|---|
| 10 | Gemma 로드·채점 함수. 프로세서가 문항별 이미지 목록을 받음 (`images=[[im], [im], ...]`) |
| 11 | `G26` 생성 — gemma-4-26B-A4B, soft token 560, TTA 2 |
| 12 | `G26B` 생성 — `G26` 복사 후 표기 변형 문항만 새 프롬프트로 덮어씀 |

**성능 — G26 (+0.005, 단일 개입 중 최대)**

MoE 라 실제 계산은 3.8B 인데 단독 0.9550 으로 학습한 8B 와 같습니다.
기여가 큰 이유는 정확도가 아니라 **구조적 비상관성**입니다 — Qwen 계열과 시각 인코더·학습 데이터가 전부 다릅니다.


**성능 — G26B (+0.0004)**

선지끼리 표기만 다른 문항(전체의 5~6%)에서 모든 뷰가 10~18%p 낮습니다.

```
Q 0.8108  Q2 0.8108  Q3 0.8108  HR 0.8108  HR2 0.7838  VC 0.7838  G26 0.8649
그 외 구간: 0.90~0.98
```

이 구간에만 "표기가 글자 단위로 일치하는 것을 고르라"는 지시를 추가하면 G26 이 0.8649 → 0.9189 로 오릅니다.
앙상블 검증 0.9717 → 0.9733, 테스트 6,714행 중 342행이 대상이고 최종 답은 6행 바뀝니다.



In [ ]:
# ============================== 10. Gemma 공통 함수 ==============================
import gc, transformers
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoProcessor
ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
assert ver >= (5, 17), f"transformers {transformers.__version__} — 1번 셀을 PHASE='gemma' 로 실행 후 세션 재시작"
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForMultimodalLM as AutoVLM

SYSTEM = ("You are a helpful visual question answering assistant. "
          "Answer using exactly one letter among a, b, c, or d. No explanation.")
G = {"id": None, "model": None, "proc": None, "ids": None}

def gemma_unload():
    for v in ["model", "processor"]:          # Qwen 이 올라와 있으면 내림
        if v in globals(): del globals()[v]
    if G["model"] is not None: G.update(id=None, model=None, proc=None, ids=None)
    gc.collect(); torch.cuda.empty_cache()

def gemma_load(model_id, soft_tokens):
    if G["id"] == (model_id, soft_tokens): return
    gemma_unload(); t0 = time.time()
    proc = AutoProcessor.from_pretrained(model_id, padding_side="left")
    tok = proc.tokenizer
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    ip, applied = proc.image_processor, {}
    for k in ["max_soft_tokens", "max_num_tokens", "num_soft_tokens", "soft_tokens", "max_pixels"]:
        if hasattr(ip, k): setattr(ip, k, int(soft_tokens)); applied[k] = int(soft_tokens)
    assert applied, f"토큰 예산 속성을 찾지 못했습니다: {sorted(vars(ip))[:40]}"
    m = AutoVLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map={"": 0},
                                attn_implementation="sdpa")
    m.config.use_cache = False; m.eval()
    ids = [tok.encode(c, add_special_tokens=False) for c in CH]
    assert all(len(t) == 1 for t in ids), f"a/b/c/d 가 1토큰이 아닙니다: {ids}"
    G.update(id=(model_id, soft_tokens), model=m, proc=proc, ids=[t[0] for t in ids])
    print(f"{model_id} 로드 {time.time()-t0:.0f}초 | 토큰 예산 {applied} | "
          f"VRAM {torch.cuda.memory_allocated()/1024**3:.1f} GB")

def gtext(row, perm):                          # 프롬프트 형식은 Qwen 뷰와 동일
    opts  = [str(row[c]).strip() for c in CH]
    shown = [opts[perm[j]] for j in range(4)]
    user = (f"{str(row['question']).strip()}\n"
            f"(a) {shown[0]}\n(b) {shown[1]}\n(c) {shown[2]}\n(d) {shown[3]}\n\n"
            "정답을 a, b, c, d 중 소문자 한 글자로만 출력하세요.")
    msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user}]}]
    return G["proc"].apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def gimg(path):
    with Image.open(path) as im:               # 크기는 Gemma 프로세서가 토큰 예산에 맞춰 처리
        return im.convert("RGB")

@torch.no_grad()
def gpredict(df, n_tta=2, desc="predict", save_path=None, batch=8, workers=4):
    probs = np.zeros((len(df), 4), np.float32); done = np.zeros(len(df), bool)
    if save_path and os.path.exists(save_path):
        z = np.load(save_path); probs, done = z["probs"], z["done"]
        print(f"이어서 진행: {int(done.sum())}/{len(df)}")
    idx = [i for i in range(len(df)) if not done[i]]
    batches = [idx[i:i + batch] for i in range(0, len(idx), batch)]
    if not batches: return probs
    prep = lambda bi: (bi, [df.iloc[j] for j in bi], [gimg(df.iloc[j]["path"]) for j in bi])
    t0 = time.time(); seen = 0
    with ThreadPoolExecutor(workers) as ex:    # GPU 계산 중에 다음 배치 사진을 미리 연다
        pending = [ex.submit(prep, b) for b in batches[:workers]]; nxt = len(pending)
        for k in tqdm(range(len(batches)), desc=desc):
            bi, rows, ims = pending.pop(0).result()
            if nxt < len(batches): pending.append(ex.submit(prep, batches[nxt])); nxt += 1
            agg = torch.zeros(len(rows), 4, device="cuda")
            for s in [0, 2, 1, 3][:n_tta]:
                perm = [(j + s) % 4 for j in range(4)]
                # Gemma 프로세서는 문항마다 이미지 목록을 받는다: [[img], [img], ...]
                enc = G["proc"](text=[gtext(r, perm) for r in rows], images=[[im] for im in ims],
                                padding=True, return_tensors="pt").to("cuda")
                lg = G["model"](**enc, use_cache=False, logits_to_keep=1).logits[:, -1, G["ids"]].float()
                agg[:, perm] += F.log_softmax(lg, -1)      # 화면 자리 점수 → 원래 보기 자리
            probs[bi] = F.softmax(agg / n_tta, -1).cpu().numpy(); done[bi] = True; seen += len(bi)
            for im in ims: im.close()
            if k == 0:
                sec = (time.time() - t0) / len(bi)
                print(f"  첫 배치 {sec:.2f}초·장 → {len(df)}장 예상 {sec*len(df)/3600:.1f}시간")
            if save_path and (k % 20 == 19 or k == len(batches) - 1):
                np.savez(save_path, probs=probs, done=done)
    print(f"  {seen}장 / {(time.time()-t0)/60:.1f}분")
    return probs

def gemma_view(model_id, tag, soft_tokens=1120, n_tta=2, batch=8, do_test=True):
    gemma_load(model_id, soft_tokens)
    va_g = gpredict(va_df, n_tta=n_tta, desc=f"valid {tag}", batch=batch)
    print(f"[{tag}] 검증 정확도 {acc_of(va_g, y_va):.4f}")
    np.save(f"{OUT_DIR}/valprobs_{tag}.npy", va_g)
    for k in ["Q", "HR", "HR2", "Q2", "Q3"]:
        f = f"{OUT_DIR}/valprobs_{k}.npy"
        if os.path.exists(f):
            v = np.load(f)
            print(f"  {k:4s} {acc_of(v, y_va):.4f} | {k}가 틀리고 {tag}가 맞힌 문항 "
                  f"{int(((v.argmax(1) != y_va) & (va_g.argmax(1) == y_va)).sum())}")
    if not do_test:
        print("do_test=False → test 는 건너뜁니다. 검증 결과를 보고 다시 실행하세요.")
        return va_g, None
    te_g = gpredict(test_df, n_tta=n_tta, desc=f"test {tag}", batch=batch,
                    save_path=f"{OUT_DIR}/testpartial_{tag}.npz")
    np.save(f"{OUT_DIR}/testprobs_{tag}.npy", te_g)
    print("저장 완료:", f"{OUT_DIR}/valprobs_{tag}.npy", f"{OUT_DIR}/testprobs_{tag}.npy")
    return va_g, te_g

print("Gemma 함수 준비 완료")

In [ ]:
# ============================== 11. Gemma 26B-A4B 뷰 ==============================
va_G26, te_G26 = gemma_view("google/gemma-4-26B-A4B-it", tag="G26",
                            soft_tokens=560, n_tta=2, batch=8, do_test=True)

In [ ]:
# ============================== 12. G26B — 표기 변형 프롬프트 (리더보드 +0.0004) ==============================
# 11번(G26)을 먼저 끝내고 실행합니다. G26 결과를 복사한 뒤, "표기 변형 문항"만 새 프롬프트로 덮어씁니다.
#
# 근거: 선지끼리 표기만 다른 문항(전체의 5~6%)에서 모든 뷰가 10~18%p 무너집니다.
#       G26 단독 0.8649 → 0.9189, 앙상블 검증 0.9717 → 0.9733, 리더보드 0.97100 → 0.97140
# 주의: LoRA 학습한 Qwen 뷰에는 이 프롬프트가 전혀 먹히지 않습니다(고침 0 망침 0). 무학습 뷰에만 씁니다.
G26B_ID, G26B_SOFT, G26B_BATCH, G26B_TTA = "google/gemma-4-26B-A4B-it", 560, 8, 2

import os, time, numpy as np, torch, torch.nn.functional as F
from PIL import Image

# ---- 표기 변형 문항 판별: 어떤 선지가 다른 선지의 부분문자열이면 변형 ----
def _nrm(s): return "".join(str(s).split()).lower()

def is_variant(row):
    o = [_nrm(row[c]) for c in CH]
    return any(i != j and len(o[i]) >= 2 and o[i] in o[j] for i in range(4) for j in range(4))

vmask = va_df.apply(is_variant, axis=1).values
tmask = test_df.apply(is_variant, axis=1).values
print(f"변형 문항  검증 {int(vmask.sum())}/{len(va_df)} | 테스트 {int(tmask.sum())}/{len(test_df)}")

# ---- 프롬프트 A(기존) / B(표기 일치 지시) ----
SYS_A = ("You are a helpful visual question answering assistant. "
         "Answer using exactly one letter among a, b, c, or d. No explanation.")
SYS_B = (SYS_A + " The choices may differ only in exact wording. "
         "Pick the one that matches the text in the image character for character. "
         "Do not add or remove any word, even if the meaning is the same.")
TAIL_B = ("보기들은 표기만 다를 수 있습니다. 사진에 적힌 글자와 완전히 일치하는 것을 고르세요.\n"
          "뜻이 같아도 단어를 더하거나 빼면 오답입니다.\n"
          "정답을 a, b, c, d 중 소문자 한 글자로만 출력하세요.")

def gtext_b(row, perm):
    o = [str(row[c]).strip() for c in CH]
    s = [o[perm[j]] for j in range(4)]
    user = (f"{str(row['question']).strip()}\n"
            f"(a) {s[0]}\n(b) {s[1]}\n(c) {s[2]}\n(d) {s[3]}\n\n{TAIL_B}")
    msgs = [{"role": "system", "content": [{"type": "text", "text": SYS_B}]},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user}]}]
    return G["proc"].apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def gpredict_b(df, n_tta=G26B_TTA, batch=G26B_BATCH, desc=""):
    out = np.zeros((len(df), 4), np.float32)
    for st in range(0, len(df), batch):
        rows = [df.iloc[i] for i in range(st, min(st + batch, len(df)))]
        ims = [Image.open(r["path"]).convert("RGB") for r in rows]
        agg = torch.zeros(len(rows), 4, device="cuda")
        for s in [0, 2][:n_tta]:
            perm = [(j + s) % 4 for j in range(4)]
            enc = G["proc"](text=[gtext_b(r, perm) for r in rows],
                            images=[[im] for im in ims],
                            padding=True, return_tensors="pt").to("cuda")
            lgt = G["model"](**enc, use_cache=False,
                             logits_to_keep=1).logits[:, -1, G["ids"]].float()
            agg[:, perm] += F.log_softmax(lgt, -1)          # 화면 자리 → 원래 보기 자리
            del enc
        out[st:st + len(rows)] = F.softmax(agg / n_tta, -1).cpu().numpy()
        for im in ims: im.close()
    return out

gemma_load(G26B_ID, G26B_SOFT)

# ---- 검증: 변형 37행만 다시 채점하고 원본에 이식 ----
vg = np.load(f"{OUT_DIR}/valprobs_G26.npy").copy()
old = vg[vmask].argmax(1)
pv = gpredict_b(va_df[vmask].reset_index(drop=True), desc="valid")
ys = y_va[vmask]
print(f"\nG26 변형구간 {(old == ys).mean():.4f} → {(pv.argmax(1) == ys).mean():.4f} "
      f"| 고침 {int(((old != ys) & (pv.argmax(1) == ys)).sum())} "
      f"망침 {int(((old == ys) & (pv.argmax(1) != ys)).sum())}")
vg[vmask] = pv
np.save(f"{OUT_DIR}/valprobs_G26B.npy", vg)
print(f"G26B 전체 검증 {acc_of(vg, y_va):.4f}  (G26 {acc_of(np.load(f'{OUT_DIR}/valprobs_G26.npy'), y_va):.4f})")

# ---- 테스트 342행 ----
t0 = time.time()
tg = np.load(f"{OUT_DIR}/testprobs_G26.npy").copy()
oldt = tg[tmask].argmax(1)
pt = gpredict_b(test_df[tmask].reset_index(drop=True), desc="test")
print(f"\n테스트 {int(tmask.sum())}행 {(time.time()-t0)/60:.1f}분 | "
      f"예측 바뀐 행 {int((oldt != pt.argmax(1)).sum())}")
tg[tmask] = pt
np.save(f"{OUT_DIR}/testprobs_G26B.npy", tg)
print("저장: valprobs_G26B.npy, testprobs_G26B.npy")

---
# C. VARCO-VISION 뷰 — `PHASE = "varco"` (학습 없음)

LLaVA-OneVision 구조에 SigLIP2 + Qwen3-14B 를 얹은 한국어 특화 VLM.

| 셀 | 내용 |
|---|---|
| 13 | AnyRes 격자를 기본 타일의 2배까지로 제한하고 뷰 생성. `VC_MODE="probe"` 로 두면 프롬프트 형식 4종 비교 |


In [ ]:
# ============================== 13. VARCO-VISION 2.0 14B 뷰 (학습 없음) ==============================
# PHASE="varco" 로 1→2→3→4 실행 후 사용. 프롬프트 형식 4종 비교(probe) 기능 포함.
# 300행 probe 실측: A 0.8867 | C 0.8800 | B 0.8567 | D 0.8433 → A 채택
VC_MODEL  = "NCSOFT/VARCO-VISION-2.0-14B"
VC_MODE   = "A"       # "probe" = 형식 4개 비교 / "A" "B" "C" "D" = 그 형식으로 뷰 생성
VC_TAG    = "VC"      # None 이면 f"VC_{VC_MODE}". 기존 VC 파일을 덮어쓰지 않으려면 다른 이름
VC_TILE   = 2         # AnyRes 격자 배수. 0 = 제한 없음
VC_TTA    = 1
VC_BATCH  = 8
VC_TEST   = True
VC_WORKERS = 4
PROBE_N   = 300

import gc, os, time
import numpy as np, torch, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as _AutoVLM
except ImportError:
    from transformers import LlavaOnevisionForConditionalGeneration as _AutoVLM

VST = globals().get("VST", {"id": None, "model": None, "proc": None, "ids": None})

def vst_load(name=VC_MODEL, tile=VC_TILE):
    if VST["id"] == (name, tile) and VST["model"] is not None: return
    VST.update(id=None, model=None, proc=None, ids=None); free_gpu()
    t0 = time.time()
    pr = AutoProcessor.from_pretrained(name)
    tk = pr.tokenizer; tk.padding_side = "left"
    if tk.pad_token_id is None: tk.pad_token = tk.eos_token
    ip = pr.image_processor
    base = getattr(ip, "size", None)
    base = (base.get("height") or base.get("shortest_edge")) if isinstance(base, dict) else 384
    m = _AutoVLM.from_pretrained(name, dtype=torch.bfloat16, device_map={"": 0},
                                 attn_implementation="sdpa")
    pin = getattr(ip, "image_grid_pinpoints", None)
    if pin and tile > 0:   # 프로세서와 모델 config 양쪽에 같은 격자를 넣어야 함
        kept = [p for p in pin if max(p) <= base * tile] or [[base, base]]
        ip.image_grid_pinpoints = kept
        m.config.image_grid_pinpoints = kept
        if hasattr(m, "model") and hasattr(m.model, "config"):
            m.model.config.image_grid_pinpoints = kept
        print(f"AnyRes 격자: 기본 {base}px | 후보 {len(pin)}개 → {len(kept)}개")
    m.config.use_cache = False; m.eval()
    ids = [tk.encode(c, add_special_tokens=False) for c in CH]
    assert all(len(t) == 1 for t in ids), f"a/b/c/d 1토큰 아님: {ids}"
    VST.update(id=(name, tile), model=m, proc=pr, ids=[t[0] for t in ids])
    print(f"로드 {time.time()-t0:.0f}초 | VRAM {torch.cuda.memory_allocated()/1024**3:.1f} GB")

SYS_KO = ("You are a helpful visual question answering assistant. "
          "Answer using exactly one letter among a, b, c, or d. No explanation.")

def vst_text(row, perm, mode):
    p = VST["proc"]
    opts = [str(row[c]).strip() for c in CH]
    s = [opts[perm[j]] for j in range(4)]
    q = str(row["question"]).strip()
    if mode == "A":      # system + (a) — probe 우승
        user = f"{q}\n(a) {s[0]}\n(b) {s[1]}\n(c) {s[2]}\n(d) {s[3]}\n\n정답을 a, b, c, d 중 소문자 한 글자로만 출력하세요."
        msgs = [{"role":"system","content":[{"type":"text","text":SYS_KO}]},
                {"role":"user","content":[{"type":"image"},{"type":"text","text":user}]}]
    elif mode == "B":    # system 제거
        user = f"{q}\na. {s[0]}\nb. {s[1]}\nc. {s[2]}\nd. {s[3]}\n\n정답: "
        msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":user}]}]
    elif mode == "C":    # 영어 지시 + A.
        user = (f"{q}\nA. {s[0]}\nB. {s[1]}\nC. {s[2]}\nD. {s[3]}\n\n"
                "Answer with a single lowercase letter: a, b, c, or d.")
        msgs = [{"role":"system","content":[{"type":"text","text":SYS_KO}]},
                {"role":"user","content":[{"type":"image"},{"type":"text","text":user}]}]
    else:                # D: 텍스트 먼저, 이미지 뒤
        user = f"{q}\n(a) {s[0]}\n(b) {s[1]}\n(c) {s[2]}\n(d) {s[3]}"
        msgs = [{"role":"user","content":[{"type":"text","text":user},{"type":"image"},
                                          {"type":"text","text":"정답:"}]}]
    return p.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _vimg(path):
    with Image.open(path) as im: return im.convert("RGB")

def _vcall(texts, ims):
    p = VST["proc"]
    try:    return p(text=texts, images=ims, padding=True, return_tensors="pt").to("cuda")
    except Exception:
        return p(text=texts, images=[[i] for i in ims], padding=True, return_tensors="pt").to("cuda")

def _vlogits(enc):
    m = VST["model"]
    try:    out = m(**enc, use_cache=False, logits_to_keep=1)
    except TypeError: out = m(**enc, use_cache=False)
    return out.logits[:, -1, VST["ids"]].float()

@torch.no_grad()
def vst_predict(df, mode, n_tta=VC_TTA, desc="predict", save_path=None, batch=VC_BATCH):
    probs = np.zeros((len(df), 4), np.float32); done = np.zeros(len(df), bool)
    if save_path and os.path.exists(save_path):
        z = np.load(save_path); probs, done = z["probs"], z["done"]
        print(f"이어서: {int(done.sum())}/{len(df)}")
    idx = [i for i in range(len(df)) if not done[i]]
    batches = [idx[i:i+batch] for i in range(0, len(idx), batch)]
    if not batches: return probs
    prep = lambda bi: (bi, [df.iloc[j] for j in bi], [_vimg(df.iloc[j]["path"]) for j in bi])
    t0 = time.time()
    with ThreadPoolExecutor(VC_WORKERS) as ex:
        pend = [ex.submit(prep, b) for b in batches[:VC_WORKERS]]; nxt = len(pend)
        for k in tqdm(range(len(batches)), desc=desc):
            bi, rows, ims = pend.pop(0).result()
            if nxt < len(batches): pend.append(ex.submit(prep, batches[nxt])); nxt += 1
            agg = torch.zeros(len(rows), 4, device="cuda")
            for s in [0, 2, 1, 3][:n_tta]:
                perm = [(j + s) % 4 for j in range(4)]
                enc = _vcall([vst_text(r, perm, mode) for r in rows], ims)
                if k == 0 and s == 0: print(f"  입력 {enc['input_ids'].shape[1]}토큰/문항")
                agg[:, perm] += F.log_softmax(_vlogits(enc), -1)   # 화면 자리 → 원래 보기 자리
                del enc
            probs[bi] = F.softmax(agg / n_tta, -1).cpu().numpy(); done[bi] = True
            for im in ims: im.close()
            del agg
            if k == 0:
                sec = (time.time() - t0) / len(bi)
                print(f"  {sec:.2f}초·장 → 검증600 {sec*600/60:.0f}분 | 테스트 {sec*len(test_df)/3600:.1f}시간")
            if k % 20 == 19: torch.cuda.empty_cache()
            if save_path and (k % 20 == 19 or k == len(batches) - 1):
                np.savez(save_path, probs=probs, done=done)
    print(f"  {(time.time()-t0)/60:.1f}분")
    return probs

# ---------------- 실행 ----------------
vst_load()
if VC_MODE == "probe":
    res = {}
    for md_ in ["A", "B", "C", "D"]:
        try:
            pr = vst_predict(va_df.iloc[:PROBE_N], md_, n_tta=1, desc=f"probe {md_}")
            res[md_] = float((pr.argmax(1) == y_va[:PROBE_N]).mean()); print(f"[{md_}] {res[md_]:.4f}")
        except Exception as e:
            print(f"[{md_}] 실패: {type(e).__name__} {str(e)[:160]}"); free_gpu(False)
    if res:
        b = max(res.items(), key=lambda x: x[1])
        print("\n" + " | ".join(f"{k} {v:.4f}" for k, v in res.items()))
        print(f"최고 {b[0]} {b[1]:.4f} → VC_MODE 를 '{b[0]}' 로 바꿔 다시 실행")
        print(f"※ {PROBE_N}행 표준오차 ≈ {(0.88*0.12/PROBE_N)**0.5*100:.1f}%p. 이보다 작은 차이는 우연입니다")
    else:
        print("전부 실패 → VC_BATCH 를 4로 낮추세요")
else:
    tag = VC_TAG or f"VC_{VC_MODE}"
    va_p = vst_predict(va_df, VC_MODE, desc=f"valid {tag}")
    print(f"[{tag}] 검증 {acc_of(va_p, y_va):.4f}")
    np.save(f"{OUT_DIR}/valprobs_{tag}.npy", va_p)
    for k in ["Q", "Q2", "Q3", "HR", "HR2", "G26", "IV38"]:
        f = f"{OUT_DIR}/valprobs_{k}.npy"
        if os.path.exists(f):
            v = np.load(f)
            print(f"  {k:5s} {acc_of(v, y_va):.4f} | {k} 오답·{tag} 정답 "
                  f"{int(((v.argmax(1) != y_va) & (va_p.argmax(1) == y_va)).sum())}")
    if VC_TEST:
        te_p = vst_predict(test_df, VC_MODE, desc=f"test {tag}",
                           save_path=f"{OUT_DIR}/testpartial_{tag}.npz")
        np.save(f"{OUT_DIR}/testprobs_{tag}.npy", te_p)
        print("저장:", f"{OUT_DIR}/testprobs_{tag}.npy")

---
# D. 앙상블 · 제출

| 셀 | 내용 |
|---|---|
| 14 | 뷰별 온도 보정 → 계열 안에서 로그확률 평균 → 계열끼리 1/3. `sub_best.csv` 생성 |
| 15 | 제출 파일 목록 및 내려받기 |

**온도 보정**: 각 뷰의 로그확률을 log-loss 최소화로 구한 온도 T 로 나눕니다.
학습한 Qwen 뷰는 0.75~1.04 로 이미 잘 보정돼 있고, 무학습 Gemma 는 3.36 으로 심하게 과확신합니다.
보정을 빼면 검증이 −3문항입니다.

**계열 평균**: 계열 안에서 뷰 수로 나눠 평균한 뒤 계열끼리 균등하게 섞습니다.
계열을 4개로 늘리면 Qwen 지분이 1/3 → 1/4 로 희석돼 항상 손해였고, 2개로 줄여도 손해였습니다.
**3계열이 측정으로 확인된 최적**입니다.


In [ ]:
# ============================== 14. 앙상블 · 제출 (최고점 구성) ==============================
# 뷰별 온도 보정 → 계열 안에서 로그확률 평균 → 계열끼리 1/3씩.
#
# 계열 구성 (측정으로 확정된 최적):
#   Q  = Q, Q2, Q3       (Qwen3-VL-8B  + LoRA, 해상도 768 / 1024 / 1280)
#   HR = HR, HR2, VC     (Qwen3-VL-4B  + LoRA, 1280 / 1024) + VARCO 무학습
#   G  = G26B            (Gemma 4 26B-A4B 무학습 + 표기 프롬프트)
import os, numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import binomtest

KS  = ["Q", "Q2", "Q3", "HR", "HR2", "VC", "G26B"]
FAM = {"Q": ["Q", "Q2", "Q3"], "HR": ["HR", "HR2", "VC"], "G": ["G26B"]}

miss = [k for k in KS if not (os.path.exists(f"{OUT_DIR}/valprobs_{k}.npy")
                              and os.path.exists(f"{OUT_DIR}/testprobs_{k}.npy"))]
assert not miss, f"뷰 파일이 없습니다: {miss}"

yv = np.load(f"{OUT_DIR}/val_y.npy")
VP = {k: np.load(f"{OUT_DIR}/valprobs_{k}.npy")  for k in KS}
TP = {k: np.load(f"{OUT_DIR}/testprobs_{k}.npy") for k in KS}

def sm(x):
    x = x - x.max(-1, keepdims=True); e = np.exp(x); return e / e.sum(-1, keepdims=True)
def lg(p): return np.log(np.clip(p, 1e-8, 1))
def fitT(p, y):
    L = lg(p)
    f = lambda t: -np.log(np.clip(sm(L / t)[np.arange(len(y)), y], 1e-15, 1)).mean()
    return float(minimize_scalar(f, bounds=(0.3, 5.0), method="bounded").x)

T = {k: fitT(VP[k], yv) for k in KS}
for k in KS:
    print(f"  {k:5s} 단독 {acc_of(VP[k], yv):.4f} | 온도 {T[k]:.2f}")

W = {k: 1 / len(FAM) / len(v) for f, v in FAM.items() for k in v}
mix = lambda D: sm(sum(w * lg(sm(lg(D[k]) / T[k])) for k, w in W.items()))

vm = mix(VP)
print(f"\n앙상블 검증 {acc_of(vm, yv):.4f} | 가중치 " +
      " ".join(f"{k}:{w:.3f}" for k, w in W.items()))
write_sub(mix(TP), "sub_best.csv")

# 참고: G26B 대신 원본 G26 을 썼을 때와 비교 (리더보드 0.97140 vs 0.97100)
if os.path.exists(f"{OUT_DIR}/testprobs_G26.npy"):
    VP2, TP2 = dict(VP), dict(TP)
    VP2["G26B"] = np.load(f"{OUT_DIR}/valprobs_G26.npy")
    TP2["G26B"] = np.load(f"{OUT_DIR}/testprobs_G26.npy")
    T["G26B"] = fitT(VP2["G26B"], yv)
    v2 = mix(VP2)
    a, b = vm.argmax(1) == yv, v2.argmax(1) == yv
    r, g = int((~b & a).sum()), int((b & ~a).sum())
    print(f"\n[대조] G26 원본 사용시 검증 {acc_of(v2, yv):.4f} "
          f"| G26B 가 고친 문항 {r}, 망친 문항 {g} "
          f"| p {binomtest(r, r+g, 0.5).pvalue:.3f}" if r + g else "")
    write_sub(mix(TP2), "sub_base_G26.csv")
    x, y = pd.read_csv("sub_best.csv"), pd.read_csv("sub_base_G26.csv")
    print(f"두 CSV 의 답이 다른 행: {int((x.iloc[:,1] != y.iloc[:,1]).sum())}")

In [ ]:
# ============================== 15. 제출 파일 내려받기 ==============================
import glob
from google.colab import files

subs = sorted(glob.glob("sub_*.csv"), key=os.path.getmtime, reverse=True)
print("만들어진 제출 파일 (최신순):")
for s in subs:
    print(f"  {s:28s} 드라이브 사본: {EXP_DIR}/{s}")

PICK = "sub_best.csv"      # ← 내려받을 파일 이름을 여기서 고르세요
if os.path.exists(PICK):
    print("\n내려받기:", PICK); files.download(PICK)
elif subs:
    print("\n" + PICK + " 가 없어 최신 파일을 내려받습니다:", subs[0]); files.download(subs[0])
else:
    print("\n제출 파일이 없습니다. 15번 셀을 먼저 실행하세요.")

---
# 부록 — 측정 기록

효과가 없다고 확인된 것들. 수치는 검증 600행 또는 리더보드 실측입니다.

## 모델 추가

| 시도 | 결과 |
|---|---|
| gemma-4-E4B (`G4B`) | 단독 0.9167. 넣은 구성 0.97080 vs 뺀 구성 0.97100 |
| InternVL3-38B (`IV38`) | 단독 0.9367. 지분 10/20/30/40%, VC 교체, 병행 투입 — 다섯 구성 전부 순증 음수 |
| gemma-4-31B 캐스케이드 | 순변화 +2문항. 저확신 부분집합에서 31B 가 오히려 낮음 (0.8083 vs 0.8250) |

InternVL3.5 대신 InternVL3 을 쓴 이유: 3.5 의 백본이 Qwen3-32B 로 주력과 같은 세대입니다.
비상관성을 사는 것이 목적이므로 한 세대 떨어진 Qwen2.5-32B 백본 쪽을 택했습니다.

## 입력 · 추론

| 시도 | 결과 |
|---|---|
| 해상도 2048 / 3072 | 이득 없음. 이미지가 0.7~1.1MP 라 1024토큰이 이미 원본 해상도 |
| 타일링 | 위와 같은 이유 |
| TTA 확장 | 순열 민감도 2.5%(15/600). TTA2 가 s=0 단독보다 낮음 (0.9650 vs 0.9667) |
| 글자 영역 크롭 | 글자 면적비 중앙값 0.89. 오답행 0.86 vs 정답행 0.89 — 배경 낭비가 오답 원인이 아님 |
| Qwen 뷰에 표기 프롬프트 | 고침 0 망침 0. LoRA 가 프롬프트에 고정됨 |
| VARCO 에 표기 프롬프트 | 단독은 개선되나 지분 1/9 라 앙상블 답이 0행 변경 |

## 후처리 · 대체 신호

| 시도 | 단독 정확도 | 앙상블 오답 16개 중 정답 | 결과 |
|---|---|---|---|
| 눈 가린 뷰 (이미지 없이 선지만) | 0.5167 | 4개 (무작위 4.0) | 모든 가중치 음수 |
| 선지 우도 (cloze) | 0.8533 | 4개 | 모든 가중치 음수 |
| PMI (이미지 − 텍스트) | 0.8267 | 5개 | 모든 가중치 음수 |
| 받아적기 + 문자열 매칭 | 0.8649 (변형구간) | — | 검증 +2(p=0.5), **리더보드 0.97140 → 0.97080** |
| top-2 리랭크 | — | — | 순변화 +1문항 |
| 가중치 무작위 탐색 | — | — | 리더보드 0.7050 |

**눈 가린 뷰 0.5167 의 의미**: 이미지 없이 선지만 보고 절반을 맞힙니다. LLM 이 정답에서 오답 선지를 파생시킨 흔적입니다.
다만 이미지 뷰들도 같은 선지 텍스트를 보므로 이미 그 신호를 소비하고 있고, 새로 얻을 정보가 0 입니다.
선지를 하나씩 독립으로 채점하면(우도 방식) 0.2467 로 떨어집니다 — **누출은 넷을 나란히 비교할 때만 발생합니다.**

**받아적기 실패의 교훈**: 표기 변형 문항이 검증에 37행뿐이라, 그 위에서 잰 +2/0 은 p=0.5 로 동전 던지기와 같은 증거였습니다.
테스트 342행에 적용하자 11행이 바뀌며 순증 −4 였습니다. 37행으로 342행을 추정할 수 없습니다.

## 라벨

| 시도 | 결과 |
|---|---|
| 표기 변형 문항의 정답 편향 (학습 476행) | 최장 39.1% / 최단 38.0% / 짧은형 39.3% / 긴형 37.4% — 규칙 없음 |
| 학습 라벨 정제 후 재학습 | 학습한 5뷰가 무학습 G26 보다 변형 구간에서 낮음. 학습이 이 약점을 배우지 못함 |

## 남은 오답 17개의 성격

사진까지 직접 확인한 분류입니다.

| 유형 | 개수 | 예시 |
|---|---|---|
| 라벨 오류 | 2 | 간판의 `太阳镜`(선글라스)·`太阳帽`(모자) 를 묻는데 정답이 '치마'(`民族裙`) |
| 표기 변형 | 3 | `쌀의 날` / `쌀의 날 기념` / `쌀의 날 기념일` / `쌀의 날 행사` |
| 문제 설계 모호 | 3 | 비상벨 엠블럼에 112·119 가 함께 표기됨 |
| 모델 판독 실패 | 4 | 자음만 적힌 간판(`ㅅㅈㅇㅇㄱ`), 작은 가격표, 지도상 구간 거리 |
| 검증 불가 | 5 | 셈 문제, 부정 질문 |

절반 가까이가 연산 자원이나 모델 크기와 무관합니다.
실제로 38B(InternVL3)는 학습한 8B 보다 단독 정확도가 낮았고, 31B(Gemma)는 어려운 부분집합에서 26B 보다 낮았습니다.